In [1]:
import pandas as pd
import numpy as np 
from pathlib import Path
from os.path import join, getsize
import kaldiio # for use with kaldi type format
import pyarrow as pa
from pyarrow import csv
import json 

In [2]:
path = '/om2/data/public/GigaSpeech/'

split = 'XL'
# split = 'DEV'
audio_path = Path(join(path, 'audio'))
samp_rate = 16000
manifest_path = Path(f"{path}/data/{split}.csv")

In [3]:
manifest = pd.read_csv(manifest_path, sep='\t',
                               usecols=['wav_filename', 'speaker', 'transcript']) 

In [4]:
manifest = manifest[~manifest.isna().any(axis=1)] # toss nan file here

In [5]:
manifest.isna().any()

wav_filename    False
transcript      False
speaker         False
dtype: bool

In [6]:
segments = pd.read_csv(Path(path,'data', 'segments'), header=None, 
                       names=['wav_filename', 'speaker', 'start', 'end'], sep='\t')

In [7]:
# wavscp_k = kaldiio.load_scp(Path(path,'data','wav.scp').as_posix())
wavscp = pd.read_csv(Path(path,'data','wav.scp'), header=None,
                                 names=['speaker', 'wav_path'], sep='\t',
                                 )

In [ ]:
new_man = pd.merge(manifest,segments, on=['wav_filename', 'speaker'])

In [ ]:
new_man = pd.merge(new_man, wavscp, on='speaker')

In [ ]:
new_man.isna().any()

In [ ]:
new_man.head()

In [ ]:
new_man.sort_values(by="transcript", key=lambda x: x.str.len(), inplace=True)

In [ ]:
new_man.head()

In [71]:
new_man['n_words'] = new_man['transcript'].str.split().str.len()

In [72]:
new_man.head()

,wav_filename,transcript,speaker,start,end,wav_path,n_words
7413942,AUD0000000304_S0000880,O,AUD0000000304,3931.99,3932.77,/rdma/vast-rdma/om3/data/public/GigaSpeech/aud...,1
4022145,YOU0000009259_S0000358,I,YOU0000009259,1476.61,1477.39,/rdma/vast-rdma/om3/data/public/GigaSpeech/aud...,1
3723129,YOU0000007664_S0000126,O,YOU0000007664,2044.05,2044.80,/rdma/vast-rdma/om3/data/public/GigaSpeech/aud...,1
3723128,YOU0000007664_S0000125,M,YOU0000007664,2042.22,2043.03,/rdma/vast-rdma/om3/data/public/GigaSpeech/aud...,1
3723127,YOU0000007664_S0000124,S,YOU0000007664,2040.33,2041.26,/rdma/vast-rdma/om3/data/public/GigaSpeech/aud...,1


In [73]:
new_man = new_man.reset_index(drop=True)
new_man.head()

,wav_filename,transcript,speaker,start,end,wav_path,n_words
0,AUD0000000304_S0000880,O,AUD0000000304,3931.99,3932.77,/rdma/vast-rdma/om3/data/public/GigaSpeech/aud...,1
1,YOU0000009259_S0000358,I,YOU0000009259,1476.61,1477.39,/rdma/vast-rdma/om3/data/public/GigaSpeech/aud...,1
2,YOU0000007664_S0000126,O,YOU0000007664,2044.05,2044.80,/rdma/vast-rdma/om3/data/public/GigaSpeech/aud...,1
3,YOU0000007664_S0000125,M,YOU0000007664,2042.22,2043.03,/rdma/vast-rdma/om3/data/public/GigaSpeech/aud...,1
4,YOU0000007664_S0000124,S,YOU0000007664,2040.33,2041.26,/rdma/vast-rdma/om3/data/public/GigaSpeech/aud...,1


In [74]:
new_man.index.name = 'index'

In [75]:
new_man.shape

(8282987, 7)

In [76]:
# parse_opts = csv.ParseOptions(delimiter='\t')
# table_manifest = csv.read_csv(manifest_path, parse_options=parse_opts)

In [78]:
new_man.head()

,wav_filename,transcript,speaker,start,end,wav_path,n_words
index,,,,,,,
0,AUD0000000304_S0000880,O,AUD0000000304,3931.99,3932.77,/rdma/vast-rdma/om3/data/public/GigaSpeech/aud...,1
1,YOU0000009259_S0000358,I,YOU0000009259,1476.61,1477.39,/rdma/vast-rdma/om3/data/public/GigaSpeech/aud...,1
2,YOU0000007664_S0000126,O,YOU0000007664,2044.05,2044.80,/rdma/vast-rdma/om3/data/public/GigaSpeech/aud...,1
3,YOU0000007664_S0000125,M,YOU0000007664,2042.22,2043.03,/rdma/vast-rdma/om3/data/public/GigaSpeech/aud...,1
4,YOU0000007664_S0000124,S,YOU0000007664,2040.33,2041.26,/rdma/vast-rdma/om3/data/public/GigaSpeech/aud...,1


In [77]:
out_path = Path(path,'data','XL_munged.csv')

new_man.to_csv(out_path, index=True)

In [28]:
# train_dict = new_man.to_dict(orient='records')

In [29]:
# out_path = Path(path,'data','XL_munged.json')

# with open(out_path, 'w') as fp:
#     json.dump(train_dict, fp)

In [79]:
split = 'DEV'
manifest_path = Path(f"{path}/data/{split}.csv")
dev_naminfest = pd.read_csv(manifest_path, sep='\t',
                               usecols=['wav_filename', 'speaker', 'transcript']) 

In [80]:
dev_naminfest

,wav_filename,transcript,speaker
0,POD1000000004_S0000000,HEY FRIENDS I DON'T KNOW ABOUT YOU BUT WHEN I ...,POD1000000004
1,POD1000000004_S0000001,MILLIAN BELIZALIAN IS NEW PODCAST FOR KIDS IN ...,POD1000000004
2,POD1000000004_S0000002,MILLION BELIZALIAN HELPING DOLLARS MAKE MORE S...,POD1000000004
3,POD1000000004_S0000003,LEILA GREEN IS A TRAUMA SURGEON IN HOUSTON,POD1000000004
4,POD1000000004_S0000004,I MEET PEOPLE ON THE WORST DAY OF THEIR LIFE U...,POD1000000004
...,...,...,...
5710,YOU1000000044_S0001772,YOU GUY HAVE A GREAT NIGHT OKAY AND THANK YOU ...,YOU1000000044
5711,YOU1000000044_S0001773,AH WE APPRECIATE YOU WE APPRECIATE YOUR SUPPOR...,YOU1000000044
5712,YOU1000000044_S0001774,AND LET'S DO IT AGAIN REAL SOON,YOU1000000044
5713,YOU1000000044_S0001775,LET'S GET IN TOUCH BYE,YOU1000000044


In [81]:
dev_man = pd.merge(dev_naminfest,segments, on=['wav_filename', 'speaker'])

In [82]:
dev_man = pd.merge(dev_man, wavscp, on='speaker')

In [83]:
# dev_dict = dev_man.to_dict(orient='records')

In [84]:
# dev_man.loc[0,'wav_path']

In [85]:
# out_path = Path(path,'data','DEV_munged.json')

# with open(out_path, 'w') as fp:
#     json.dump(dev_dict, fp)

In [86]:
dev_man['n_words'] = dev_man['transcript'].str.split().str.len()

In [87]:
dev_man.index.name = 'index'

In [88]:
dev_man.head()

,wav_filename,transcript,speaker,start,end,wav_path,n_words
index,,,,,,,
0,POD1000000004_S0000000,HEY FRIENDS I DON'T KNOW ABOUT YOU BUT WHEN I ...,POD1000000004,0.000,10.197,/rdma/vast-rdma/om3/data/public/GigaSpeech/aud...,35
1,POD1000000004_S0000001,MILLIAN BELIZALIAN IS NEW PODCAST FOR KIDS IN ...,POD1000000004,10.197,22.689,/rdma/vast-rdma/om3/data/public/GigaSpeech/aud...,41
2,POD1000000004_S0000002,MILLION BELIZALIAN HELPING DOLLARS MAKE MORE S...,POD1000000004,22.689,33.514,/rdma/vast-rdma/om3/data/public/GigaSpeech/aud...,28
3,POD1000000004_S0000003,LEILA GREEN IS A TRAUMA SURGEON IN HOUSTON,POD1000000004,33.514,36.764,/rdma/vast-rdma/om3/data/public/GigaSpeech/aud...,8
4,POD1000000004_S0000004,I MEET PEOPLE ON THE WORST DAY OF THEIR LIFE U...,POD1000000004,36.764,41.256,/rdma/vast-rdma/om3/data/public/GigaSpeech/aud...,11


In [89]:
out_path = Path(path,'data','DEV_munged.csv')

dev_man.to_csv(out_path, index=True)